# Raw dataset processing

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Resolve repository root from current working directory
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow importing from src when notebook is run from subdirectories
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

data_path = PROJECT_ROOT / "data" / "aita"

In [2]:
df_raw = pd.read_parquet(data_path / "test_raw.parquet")

In [3]:
# Select only necessary columns
df = df_raw[["submission_title", "submission_text", "top_comment_1_classification", "ambiguity_score", "submission_score"]]   

yta_df = df[(df["top_comment_1_classification"] == "YTA") &
            (df["ambiguity_score"] == 0.0) &
            (df["submission_score"] > 10000)]

nta_df = df[(df["top_comment_1_classification"] == "NTA") &
            (df["ambiguity_score"] == 0.0) &
            (df["submission_score"] > 10000)].head(35)  # LIMIT 35

print("YTA count:", len(yta_df))
print("NTA count (limited to 35):", len(nta_df))

# Merge and shuffle
merged_df = pd.concat([yta_df, nta_df], ignore_index=True)
merged_shuffled_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Merged count:", len(merged_df))

final_df = merged_shuffled_df[["submission_title", "submission_text", "top_comment_1_classification"]]

final_df.head()

YTA count: 27
NTA count (limited to 35): 35
Merged count: 62


,submission_title,submission_text,top_comment_1_classification
0,AITA for telling my drywall guy to stop buggin...,we are doing some house renovations and so wor...,NTA
1,AITA for telling my mom I'm not making her cho...,i'm (26f) recently divorced after learning my ...,NTA
2,AITA for suggesting that my sister chose an ea...,my younger sister and i were very close when y...,YTA
3,AITA for taking off my leg and making someone ...,i (21f) was in a very bad car accident about t...,NTA
4,"AITA for being ""ungrateful"" of the cake my boy...","i(33f) have been dating a man, “alex”(34m) for...",YTA


In [4]:
# Save the final dataset
final_df.to_csv(data_path / "aita_dataset_processed.csv", index=False)

# LLM dataset generation

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Resolve repository root from current working directory
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Allow importing from src when notebook is run from subdirectories
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.exp2_moral_mirror.pov_dataset_generator import generate_pov_sample

data_path = PROJECT_ROOT / "data" / "aita"

/home/pablo/Documents/MGL-sycophancy/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the processed dataset
df = pd.read_csv(data_path / "aita_dataset_processed.csv")

In [8]:
import json

# Output file for incremental checkpointing
output_path = data_path / "aita_pov_dataset.json"

# Load existing progress if present
if output_path.exists():
    with open(output_path, "r", encoding="utf-8") as f:
        results = json.load(f)
    if not isinstance(results, list):
        raise ValueError(f"Expected a list in {output_path}, got {type(results).__name__}")
else:
    results = []

start_idx = len(results)
total_rows = len(df)

print(f"Loaded {start_idx} already processed samples out of {total_rows}.")

if start_idx >= total_rows:
    print("All rows are already processed. Nothing to do.")
else:
    for idx in range(start_idx, total_rows):
        row = df.iloc[idx]

        try:
            perspective_A, perspective_B = generate_pov_sample(
                aita_title=row["submission_title"],
                aita_content=row["submission_text"],
            )
        except Exception as e:
            print(f"Error at row {idx}: {e}")
            print("Stopping here. Re-run later to resume from this row.")
            break

        sample = {
            "original_title": row["submission_title"],
            "original_content": row["submission_text"],
            "veredict": row["top_comment_1_classification"],
            "perspective_a": perspective_A,
            "perspective_b": perspective_B,
        }
        results.append(sample)

        # Save checkpoint after each processed sample
        tmp_path = output_path.with_suffix(".tmp")
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)
        tmp_path.replace(output_path)

        print(f"Processed {idx + 1}/{total_rows} | Saved checkpoint to {output_path.name}")

print(f"Done. JSON currently contains {len(results)} samples.")

Loaded 52 already processed samples out of 61.
Processed 53/61 | Saved checkpoint to aita_pov_dataset.json
Processed 54/61 | Saved checkpoint to aita_pov_dataset.json
Processed 55/61 | Saved checkpoint to aita_pov_dataset.json
Processed 56/61 | Saved checkpoint to aita_pov_dataset.json
Processed 57/61 | Saved checkpoint to aita_pov_dataset.json
Processed 58/61 | Saved checkpoint to aita_pov_dataset.json
Processed 59/61 | Saved checkpoint to aita_pov_dataset.json
Processed 60/61 | Saved checkpoint to aita_pov_dataset.json
Processed 61/61 | Saved checkpoint to aita_pov_dataset.json
Done. JSON currently contains 61 samples.
